In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz


In [0]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [0]:
df_base_score_bureau_movel = spark.read.parquet("/Volumes/hackathon2025/bronze/base_score_bureau_movel/")

display(df_base_score_bureau_movel.limit(30))

In [0]:
df_base_score_bureau_movel.createOrReplaceTempView("raw_00")

In [0]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM raw_00
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

display(contagem_safras)

In [0]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM raw_00
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM raw_00 r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [0]:
df_resultado = contagem_percentual("FLAG_INSTALACAO")
display(df_resultado)

In [0]:
df_resultado = contagem_percentual("PROD")
display(df_resultado)

In [0]:
df_resultado = contagem_percentual("flag_mig2")
display(df_resultado)

In [0]:
print('lista de colunas para tipar')
for col in spark.table("raw_00").columns:
    print('try_cast(' + col + ' as) as ' + col + ',')

In [0]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            cast(NUM_CPF as string) as NUM_CPF,
            cast(SAFRA as int) as SAFRA,
            cast(FLAG_INSTALACAO as int) as FLAG_INSTALACAO,
            cast(FPD as int) as FPD,
            cast(PROD as string) as PROD,
            cast(flag_mig2 as string) as flag_mig2,
            cast(SCORE_01 as int) as SCORE_01,
            cast(SCORE_02 as int) as SCORE_02,
            {pdthproc} as DATPROC

        from
            raw_00
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.count()  

In [0]:
lake.printSchema()

In [0]:
display(lake)

In [0]:
for col in lake.columns:
    agg_result = lake.agg(
        {col: "count"} 
    ).collect()[0]
    
    total = lake.count()
    nao_nulos = agg_result[f"count({col})"]
    nulos = total - nao_nulos
    
    if nulos > 0:
        print(f"{col}: {nulos} nulos ({nulos/total*100:.2f}%)")

In [0]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, SAFRA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.count()  

In [0]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

display(contagem_safras)

In [0]:
from delta.tables import DeltaTable

silver_table = "hackathon2025.silver.base_score_bureau_movel"

if not spark.catalog.tableExists(silver_table):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table) 
    )

    print("Tabela silver criada com sucesso")

else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forName(spark, silver_table)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.SAFRA = s.SAFRA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Dados inseridos com sucesso...")


In [0]:
name = "base_score_bureau_movel"

df_controle = spark.sql("""
    SELECT
        '{name_table}'        AS nome_tabela,
        SAFRA                 AS safra,
        COUNT(*)              AS qtd_registros,
        current_timestamp()   AS datproc
    FROM lake_dedup
    GROUP BY SAFRA
""".format(name_table=name))

display(df_controle)


In [0]:
silver_table_controle = "hackathon2025.silver.controle"
if not spark.catalog.tableExists(silver_table_controle):
    print("Tabela silver_controle não existe. Criando...")

    (
        df_controle
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table_controle)  
    )

    print("Tabela silver_controle criada com sucesso")

else:
    print("Tabela de controle existe. Inserindo novo registro...")

    delta_silver = DeltaTable.forName(spark, silver_table_controle)

    (
        df_controle
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table_controle)
    )
    print("Dados inseridos com sucesso...")